# Ingest circuits.csv file
- 0.- Define Schema
- 1.- Read the file using spark dataframe reader API
- 2.- Add Metadata Columns
-     * Source File
-     * Ingestion Timestamp
- 3.- Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers                

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

0.-Define Schema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql import functions as F


circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

Step 1 - Read the CSV file using the dataframe reader API

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', 'True')
        .schema(circuits_schema)
        #.load('abfss://formula1@eberdatabrickscoursedl.dfs.core.windows.net/landing/circuits.csv')
        .load(source_file)
)

In [0]:
display(circuits_df)

 2.- Add Metadata Columns
-     * Source File
-     * Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

display(circuits_final_df)


- 3.- Write to bronze delta table

In [0]:
(
    circuits_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)

)

In [0]:
%sql
Select * from formula1.bronze.circuits

In [0]:
display(spark.table('formula1.bronze.circuits'))